# 04 - AI adapter: certificates over LLM-style claims
Demonstrates the guardrail *adapter* (`postop.adapters`, not core PosTop) pattern: an explicit hypothesis/observation encoding step, then cover + positivity checks producing structured certificates. No network calls -- everything uses the deterministic cases from `examples/llm_guardrail.py`.

In [ ]:
from postop import IncidenceSystem, from_dict
from postop.adapters import ClaimNormalizer, PosTopGuardrail

cases = {
    'flu_case': ['flu', 'fever', 'cough', 'fatigue'],
    'covid_case': ['covid', 'fever', 'cough', 'loss_of_taste'],
    'cold_case': ['common_cold', 'cough', 'runny_nose'],
}

system = IncidenceSystem(from_dict(cases))
normalizer = ClaimNormalizer()
normalizer.register('flu', 'influenza')

# encode_hypothesis / encode_observation are explicit here -- an LLM
# hypothesis string is NOT automatically a PosTop generator; this
# demo's encoding happens to be normalizer.normalize (the default).
guardrail = PosTopGuardrail(system, normalizer=normalizer)
observed = {'fever', 'cough', 'fatigue'}
print('Observed context:', observed)
print('Hypotheses:', ['influenza', 'covid', 'common_cold'])

In [ ]:
result = guardrail.filter_hypotheses(['influenza', 'covid', 'common_cold'], observed)
print('Valid hypotheses:', result['valid_hypotheses'])
for detail in result['details']:
    print(detail.claim, detail.status, '->', detail.explanation)

In [ ]:
consistency = guardrail.check_consistency(['flu', 'covid'])
print('Consistency check:', consistency)

single_check = guardrail.check_claim('tuberculosis', {'cough', 'runny_nose'})
print('Single claim evaluation:', single_check)

In [ ]:
# The same claim via the certificate-producing core API directly,
# for comparison with the guardrail's higher-level VALID/INVALID/... status.
cover_cert = system.explain_cover('flu', observed)
positivity_cert = system.explain_positive('flu', observed | {'flu'})
print('Cover certificate:', cover_cert.to_dict())
print('Positivity certificate:', positivity_cert.to_dict())